In [5]:
import pandas as pd

df_fatture = pd.read_csv("fatture_aperte.csv", parse_dates=["data_emissione", "data_scadenza"])

# Data di riferimento dell'analisi: "oggi" per l'azienda simulata
oggi = pd.Timestamp("2026-07-03")

print(f"Fatture totali: {len(df_fatture)}")
df_fatture.head()

Fatture totali: 13


,numero_fattura,cliente,importo,data_emissione,data_scadenza,numero
0,FT-2026-006,AgriToscana SOC COOP,1154.42,2026-01-07,2026-02-06,6
1,FT-2026-010,Meccanica Fiorentina SRL,946.30,2026-03-29,2026-04-28,10
2,FT-2026-012,Studio Neri & Associati,4267.97,2026-01-25,2026-02-24,12
3,FT-2026-015,Bianchi Alimentari SPA,2849.80,2026-04-07,2026-05-07,15
4,FT-2026-017,Rossi Costruzioni SRL,3374.06,2026-01-18,2026-02-17,17


In [6]:
# Giorni di ritardo rispetto alla scadenza (negativo = non ancora scaduta)
df_fatture["giorni_ritardo"] = (oggi - df_fatture["data_scadenza"]).dt.days

# Assegnazione della fascia di aging
def fascia_aging(giorni):
    if giorni <= 0:
        return "Non scaduta"
    elif giorni <= 30:
        return "0-30 giorni"
    elif giorni <= 60:
        return "31-60 giorni"
    elif giorni <= 90:
        return "61-90 giorni"
    else:
        return "Oltre 90 giorni"

df_fatture["fascia"] = df_fatture["giorni_ritardo"].apply(fascia_aging)

df_fatture[["numero_fattura", "cliente", "importo", "data_scadenza", "giorni_ritardo", "fascia"]].head(10)

,numero_fattura,cliente,importo,data_scadenza,giorni_ritardo,fascia
0,FT-2026-006,AgriToscana SOC COOP,1154.42,2026-02-06,147,Oltre 90 giorni
1,FT-2026-010,Meccanica Fiorentina SRL,946.30,2026-04-28,66,61-90 giorni
2,FT-2026-012,Studio Neri & Associati,4267.97,2026-02-24,129,Oltre 90 giorni
3,FT-2026-015,Bianchi Alimentari SPA,2849.80,2026-05-07,57,31-60 giorni
4,FT-2026-017,Rossi Costruzioni SRL,3374.06,2026-02-17,136,Oltre 90 giorni
5,FT-2026-019,Elettrodomestici Gallo SRL,1534.27,2026-02-25,128,Oltre 90 giorni
6,FT-2026-029,Toscana Impianti SNC,3346.11,2026-04-21,73,61-90 giorni
7,FT-2026-044,AgriToscana SOC COOP,3857.42,2026-04-08,86,61-90 giorni
8,FT-2026-047,Studio Neri & Associati,3860.05,2026-03-11,114,Oltre 90 giorni
9,FT-2026-048,AgriToscana SOC COOP,4608.77,2026-06-18,15,0-30 giorni


In [7]:
print(df_fatture["fascia"].value_counts())
print(f"\nRitardo minimo: {df_fatture['giorni_ritardo'].min()} giorni")
print(f"Ritardo massimo: {df_fatture['giorni_ritardo'].max()} giorni")
print(f"\nScadenza più vecchia: {df_fatture['data_scadenza'].min()}")
print(f"Scadenza più recente: {df_fatture['data_scadenza'].max()}")

fascia
Oltre 90 giorni    7
61-90 giorni       3
31-60 giorni       2
0-30 giorni        1
Name: count, dtype: int64

Ritardo minimo: 15 giorni
Ritardo massimo: 147 giorni

Scadenza più vecchia: 2026-02-06 00:00:00
Scadenza più recente: 2026-06-18 00:00:00


In [8]:
aging = df_fatture.groupby("fascia").agg(
    numero_fatture=("numero_fattura", "count"),
    importo_totale=("importo", "sum")
).reindex(["Non scaduta", "0-30 giorni", "31-60 giorni", "61-90 giorni", "Oltre 90 giorni"])

totale = aging["importo_totale"].sum()
aging["percentuale"] = (aging["importo_totale"] / totale * 100).round(1)

print(f"Crediti aperti totali: {totale:,.2f} €")
aging

Crediti aperti totali: 39,362.41 €


,numero_fatture,importo_totale,percentuale
fascia,,,
Non scaduta,NaN,NaN,NaN
0-30 giorni,1.0,4608.77,11.7
31-60 giorni,2.0,3842.41,9.8
61-90 giorni,3.0,8149.83,20.7
Oltre 90 giorni,7.0,22761.40,57.8


In [9]:
per_cliente = df_fatture.groupby("cliente").agg(
    fatture_aperte=("numero_fattura", "count"),
    esposizione=("importo", "sum"),
    ritardo_max=("giorni_ritardo", "max"),
    ritardo_medio=("giorni_ritardo", "mean")
).sort_values("esposizione", ascending=False)

per_cliente["ritardo_medio"] = per_cliente["ritardo_medio"].round(0)

print(f"Clienti con crediti aperti: {len(per_cliente)}")
per_cliente

Clienti con crediti aperti: 8


,fatture_aperte,esposizione,ritardo_max,ritardo_medio
cliente,,,,
AgriToscana SOC COOP,5,14438.49,147,76.0
Studio Neri & Associati,2,8128.02,129,122.0
InfoService SRL,1,4745.36,92,92.0
Rossi Costruzioni SRL,1,3374.06,136,136.0
Toscana Impianti SNC,1,3346.11,73,73.0
Bianchi Alimentari SPA,1,2849.80,57,57.0
Elettrodomestici Gallo SRL,1,1534.27,128,128.0
Meccanica Fiorentina SRL,1,946.30,66,66.0


In [10]:
df_tutte = pd.read_csv("fatture.csv", parse_dates=["data_emissione", "data_scadenza"])

In [11]:
pagamenti = pd.read_csv("pagamenti_effettuati.csv", parse_dates=["data_movimento"])

# Aggancio a ogni pagamento la sua fattura (per avere scadenza e cliente)
pagamenti = pagamenti.merge(
    df_tutte[["numero_fattura", "cliente", "data_scadenza"]],
    left_on="fattura_matchata", right_on="numero_fattura"
)

pagamenti["ritardo_pagamento"] = (pagamenti["data_movimento"] - pagamenti["data_scadenza"]).dt.days

pagamenti[["cliente", "fattura_matchata", "data_scadenza", "data_movimento", "ritardo_pagamento"]].head(10)

,cliente,fattura_matchata,data_scadenza,data_movimento,ritardo_pagamento
0,Bianchi Alimentari SPA,FT-2026-040,2026-02-02,2026-02-01,-1
1,Verdi Logistica SRL,FT-2026-009,2026-02-01,2026-02-01,0
2,Toscana Impianti SNC,FT-2026-026,2026-02-14,2026-02-14,0
3,Tessuti Lucchesi SPA,FT-2026-052,2026-02-21,2026-02-25,4
4,AgriToscana SOC COOP,FT-2026-003,2026-02-26,2026-02-26,0
5,Rossi Costruzioni SRL,FT-2026-035,2026-02-23,2026-03-02,7
6,InfoService SRL,FT-2026-022,2026-02-18,2026-03-04,14
7,Toscana Impianti SNC,FT-2026-031,2026-03-07,2026-03-05,-2
8,Studio Neri & Associati,FT-2026-060,2026-02-16,2026-03-07,19
9,Rossi Costruzioni SRL,FT-2026-001,2026-02-28,2026-03-15,15


In [12]:
comportamento = pagamenti.groupby("cliente").agg(
    pagamenti_fatti=("fattura_matchata", "count"),
    ritardo_medio=("ritardo_pagamento", "mean"),
    ritardo_max=("ritardo_pagamento", "max")
).round(1).sort_values("ritardo_medio", ascending=False)

comportamento

,pagamenti_fatti,ritardo_medio,ritardo_max
cliente,,,
Studio Neri & Associati,1,19.0,19
InfoService SRL,7,12.1,17
AgriToscana SOC COOP,2,10.0,20
Rossi Costruzioni SRL,4,9.8,15
Tessuti Lucchesi SPA,5,8.6,16
Verdi Logistica SRL,5,8.4,16
Meccanica Fiorentina SRL,5,8.0,18
Bianchi Alimentari SPA,7,7.7,20
Toscana Impianti SNC,6,4.5,13


In [13]:
def scoring(riga):
    if riga["ritardo_max"] > 90 and riga["esposizione"] > 5000:
        return "C - rischio"
    elif riga["ritardo_max"] > 60 or riga["esposizione"] > 5000:
        return "B - attenzione"
    else:
        return "A - ok"

per_cliente["scoring"] = per_cliente.apply(scoring, axis=1)
per_cliente.sort_values("esposizione", ascending=False)

,fatture_aperte,esposizione,ritardo_max,ritardo_medio,scoring
cliente,,,,,
AgriToscana SOC COOP,5,14438.49,147,76.0,C - rischio
Studio Neri & Associati,2,8128.02,129,122.0,C - rischio
InfoService SRL,1,4745.36,92,92.0,B - attenzione
Rossi Costruzioni SRL,1,3374.06,136,136.0,B - attenzione
Toscana Impianti SNC,1,3346.11,73,73.0,B - attenzione
Bianchi Alimentari SPA,1,2849.80,57,57.0,A - ok
Elettrodomestici Gallo SRL,1,1534.27,128,128.0,B - attenzione
Meccanica Fiorentina SRL,1,946.30,66,66.0,B - attenzione
